# ESKALATOR ACADEMY — Chapter 4
## Basic Image Processing & OCR
### Signal Acquisition

---

```
[ESKALATOR] CHAPTER 4 INITIALIZED
[ESKALATOR] PLATFORM   : Google Colab
[ESKALATOR] LIBRARIES  : Pillow · pytesseract · re
[ESKALATOR] OBJECTIVE  : Build a manual OCR pipeline from scratch
```

---

**Briefing:**

The ESKALATOR system needs to read printed ICU monitor data from photographs taken by field nurses. Before we can use the real vision model (Chapter 5), you will build the pipeline the hard way — step by step, pixel by pixel.

By the end of this chapter, you will understand *exactly* why a smarter approach is needed.

**Run all cells from top to bottom. Do not skip.**

---
## Setup — Install Dependencies

Run this cell once at the start of every session. If your Colab runtime disconnects and restarts, run this again.

In [ ]:
!sudo apt install tesseract-ocr -y -q
!pip install pytesseract pillow -q
print("[ESKALATOR] Dependencies installed. Ready.")

---
## Setup — Generate Sample ICU Image

This cell creates a synthetic ICU monitor printout image for you to work with.
Run it once — it will save `icu_sample.jpg` to your Colab session.

In [ ]:
from PIL import Image, ImageDraw

# Create a synthetic ICU monitor printout
img = Image.new("RGB", (500, 260), color=(235, 235, 235))
draw = ImageDraw.Draw(img)

# Header
draw.rectangle([(0, 0), (500, 45)], fill=(30, 30, 30))
draw.text((15, 12), "ESKALATOR ICU MONITOR — BAY 7", fill=(0, 229, 204))

# Vitals
draw.text((20, 65),  "SpO2:",       fill=(40, 40, 40))
draw.text((120, 65), "97 %",         fill=(20, 20, 20))

draw.text((20, 110), "HR:",          fill=(40, 40, 40))
draw.text((120, 110), "82 bpm",      fill=(20, 20, 20))

draw.text((20, 155), "BP:",          fill=(40, 40, 40))
draw.text((120, 155), "120 / 80",    fill=(20, 20, 20))

draw.text((20, 200), "TEMP:",        fill=(40, 40, 40))
draw.text((120, 200), "36.8 C",      fill=(20, 20, 20))

# Footer
draw.rectangle([(0, 235), (500, 260)], fill=(200, 200, 200))
draw.text((15, 241), "Patient ID: ESK-007  |  2026-04-25  08:42", fill=(80, 80, 80))

img.save("icu_sample.jpg")
display(img)
print("[ESKALATOR] icu_sample.jpg created.")

---
# Mission 1 — Image Received
**XP: 120 · Concept: Load & Inspect an Image**

```
[ESKALATOR] STATUS: New file received from Bay 7 field sensor.
[ESKALATOR] ACTION : Confirm image integrity before processing.
```

A nurse photographed a printed ICU monitor readout and uploaded it to the system. Before the pipeline can process anything, it needs to confirm the image loaded correctly.

**Your task:** Load `icu_sample.jpg`, print its size and mode, then display it.

In [ ]:
from PIL import Image

# Load the ICU printout image
img = Image.open("icu_sample.jpg")

# Print image properties
print("Size:", img.size)   # (width, height) in pixels
print("Mode:", img.mode)   # Color mode: RGB, L, RGBA, etc.

# Display the image inline
display(img)

In [ ]:
# Verification — run this after the cell above
width, height = img.size
assert width > 0 and height > 0, "Image did not load correctly"
assert img.mode in ("RGB", "L", "RGBA"), "Unexpected image mode"
print(f"[ESKALATOR] MISSION 1 COMPLETE. Image {width}x{height}px, mode={img.mode}. +120 XP")

**Catatan:**
- `.size` mengembalikan tuple `(lebar, tinggi)` dalam piksel
- Mode `"RGB"` artinya setiap piksel punya 3 nilai: Red, Green, Blue (masing-masing 0–255)
- Semua gambar digital adalah grid piksel — tidak ada yang magic

---
# Mission 2 — Isolate the Signal
**XP: 150 · Concept: Crop & Grayscale**

```
[ESKALATOR] STATUS: Full image contains visual noise — header, footer, labels.
[ESKALATOR] ACTION : Extract region of interest. Convert to grayscale.
```

The OCR engine struggles with visual clutter. Protocol requires isolating only the numeric display region before reading.

**Your task:** Crop the region containing the vital numbers, convert to grayscale, and save it.

In [ ]:
from PIL import Image

img = Image.open("icu_sample.jpg")

# Crop to the vitals region
# Format: (left, top, right, bottom) in pixels
roi = img.crop((0, 50, 500, 235))

# Convert to grayscale — OCR works better on single-channel images
roi_gray = roi.convert("L")

roi_gray.save("roi_preprocessed.jpg")
display(roi_gray)
print("ROI size:", roi_gray.size)
print("ROI mode:", roi_gray.mode)

In [ ]:
# Verification
assert roi_gray.mode == "L", "Image harus grayscale (mode 'L')"
assert roi_gray.size[0] > 0 and roi_gray.size[1] > 0
print(f"[ESKALATOR] MISSION 2 COMPLETE. ROI extracted at {roi_gray.size}. +150 XP")

**Catatan:**
- Koordinat crop: `(kiri, atas, kanan, bawah)`. Piksel `(0, 0)` ada di pojok kiri atas
- Mode `"L"` = Luminance — hanya satu nilai per piksel (terang/gelap), bukan 3 warna
- Kalau hasil crop kelihatan salah, ubah angka koordinatnya. Ini normal — itulah masalah utama OCR tradisional

---
# Mission 3 — Enhance Contrast
**XP: 150 · Concept: Binary Threshold**

```
[ESKALATOR] STATUS: Grayscale ROI still contains ambiguous mid-tone pixels.
[ESKALATOR] ACTION : Apply binary threshold. Force every pixel to black or white.
```

The numbers bleed into the background. Clinical standard: binary thresholding — force every pixel to pure black or pure white. A digit is either there or it is not.

**Your task:** Boost contrast, then apply binary threshold to the grayscale ROI.

In [ ]:
from PIL import Image, ImageEnhance

roi_gray = Image.open("roi_preprocessed.jpg")

# Step 1: Boost contrast
enhancer = ImageEnhance.Contrast(roi_gray)
high_contrast = enhancer.enhance(2.5)   # factor > 1 = more contrast

# Step 2: Binary threshold
# Pixels above 128 → white (255)
# Pixels at or below 128 → black (0)
threshold = 128
binary = high_contrast.point(lambda px: 255 if px > threshold else 0)

binary.save("roi_binary.jpg")
display(binary)

**Kenapa ini penting?**

OCR membaca teks dengan mendeteksi piksel gelap di atas latar terang. Kalau background-nya abu-abu (nilai 180) dan teksnya abu gelap (nilai 100), OCR mungkin gagal membacanya.

Binary thresholding menghilangkan ambiguitas — hanya ada hitam dan putih.

> **Masalahnya:** nilai threshold `128` adalah **tebakan**. Di foto lain dengan pencahayaan berbeda, angka ini mungkin salah. Inilah kelemahan utama OCR tradisional.

In [ ]:
# Verification
pixels = list(binary.getdata())
unique_values = set(pixels)
assert unique_values.issubset({0, 255}), "Binary image harus hanya punya nilai piksel 0 atau 255"
print(f"[ESKALATOR] MISSION 3 COMPLETE. Unique pixel values: {unique_values}. +150 XP")

---
# Mission 4 — Run the OCR Engine
**XP: 200 · Concept: Extract Raw Text with pytesseract**

```
[ESKALATOR] STATUS: Image preprocessed. Ready for text extraction.
[ESKALATOR] ACTION : Invoke OCR engine. Log raw output exactly as received.
```

The image is prepared. Now invoke pytesseract. The output will be raw — messy, imperfect, sometimes wrong. That is expected. Your job is to observe and log it.

**Your task:** Run `image_to_string()` on the binary image and inspect the output.

In [ ]:
import pytesseract
from PIL import Image

binary_img = Image.open("roi_binary.jpg")

# Run OCR — extract all text from the image
# --psm 6 = assume a single uniform block of text
raw_text = pytesseract.image_to_string(binary_img, config='--psm 6')

print("=== RAW OCR OUTPUT ===")
print(raw_text)
print("=== END ===")

# repr() shows hidden characters like \n (newline)
print("\nRepr view:")
print(repr(raw_text))

In [ ]:
# OBSERVE: Perhatikan output di atas.
# Tulis observasimu di sini sebagai komentar:

# OCR berhasil baca: ...
# OCR salah baca / bingung: ...
# Karakter yang sering tertukar: (contoh umum: 0 vs O, 1 vs l, 8 vs B)
# Masalah lain yang kamu lihat: ...

print(f"[ESKALATOR] MISSION 4 COMPLETE. Raw OCR output logged. +200 XP")

---
# Mission 5 — Parse the Vitals
**XP: 200 · Concept: Extract Structured Data from Raw Text**

```
[ESKALATOR] STATUS: Raw OCR text received. Cannot be passed to clinical database as-is.
[ESKALATOR] ACTION : Parse text. Extract SpO2 and heart_rate as structured dict.
```

Raw text is not data. Write a parser that extracts SpO₂ and heart rate from the noisy OCR output and returns them as a structured dict. If the parser cannot find a value, it must return `None` — not crash.

**Your task:** Complete the `parse_vitals()` function.

In [ ]:
import re

def parse_vitals(text):
    """
    Given raw OCR text, extract SpO2 and heart_rate.
    Returns: {"spo2": int or None, "heart_rate": int or None}
    """
    # Find all sequences of digits in the text
    numbers = re.findall(r'\d+', text)
    
    # Convert string digits to integers
    numbers = [int(n) for n in numbers]
    
    print("Numbers found by regex:", numbers)
    
    # Extract based on physiological ranges
    # SpO2: 2-digit number between 85–100
    # Heart rate: 2-3 digit number between 40–180
    spo2 = None
    heart_rate = None
    
    for n in numbers:
        if 85 <= n <= 100 and spo2 is None:
            spo2 = n
        elif 40 <= n <= 180 and heart_rate is None:
            heart_rate = n
    
    return {"spo2": spo2, "heart_rate": heart_rate}


# Test with the actual OCR output from Mission 4
result = parse_vitals(raw_text)

print("\n=== PARSED VITALS ===")
print(result)

In [ ]:
# Verification
assert isinstance(result, dict), "parse_vitals harus return dict"
assert "spo2" in result, "dict harus punya key 'spo2'"
assert "heart_rate" in result, "dict harus punya key 'heart_rate'"

if result["spo2"] is not None:
    assert isinstance(result["spo2"], int), "spo2 harus int atau None"
if result["heart_rate"] is not None:
    assert isinstance(result["heart_rate"], int), "heart_rate harus int atau None"

print("[ESKALATOR] PIPELINE COMPLETE.")
print(f"  SpO2       : {result['spo2']} %")
print(f"  Heart Rate : {result['heart_rate']} bpm")
print("[ESKALATOR] MISSION 5 COMPLETE. +200 XP")

---
## System Log — Pipeline Audit

```
=== ESKALATOR SYSTEM LOG ===
Chapter 4 pipeline complete. Reviewing performance...

PROBLEM 1: Crop coordinates disetel manual.
  → roi = img.crop((0, 50, 500, 235))
  → Foto beda = koordinat beda = harus diubah lagi.

PROBLEM 2: Threshold value adalah tebakan.
  → threshold = 128
  → Foto gelap? Foto terlalu terang? Angka ini salah.

PROBLEM 3: Regex parser pakai asumsi range fisiologis.
  → if 85 <= n <= 100  ← ini bisa salah kalau ada angka lain di gambar
  → Tidak fleksibel untuk format printout yang berbeda.

SUMMARY:
  Manual decisions made  : ~5
  Lines of fragile code  : ~40
  Accuracy               : tidak konsisten

NEXT: Chapter 5 — Florence-2 Pipeline
  Foto yang SAMA akan dikirim ke Florence-2.
  Query: "What is the SpO2 reading?"
  Output: jawaban langsung — tanpa crop, tanpa threshold, tanpa regex.
===
```

---
## Chapter 4 Complete

**Total XP earned: 820 XP**

| Mission | Title | XP |
|---|---|---|
| M1 | Image Received | 120 |
| M2 | Isolate the Signal | 150 |
| M3 | Enhance Contrast | 150 |
| M4 | Run the OCR Engine | 200 |
| M5 | Parse the Vitals | 200 |

**Apa yang sudah kamu bangun:**
Sebuah pipeline lengkap — dari foto mentah sampai dict `{"spo2": ..., "heart_rate": ...}` — menggunakan hanya PIL, pytesseract, dan regex.

**Langkah selanjutnya:**
1. Screenshot output cell terakhir (Mission 5 verification)
2. Kirim ke mentor untuk konfirmasi Chapter 4 complete
3. Chapter 5 akan unlock: **Florence-2 Pipeline**

---
*ESKALATOR ACADEMY · Chapter 4 · Signal Acquisition*